# 🎬 Wikidata Cinema → Text Generation with Qwen (DashScope)

Pipeline complet : sous-graphe Wikidata → génération de **paragraphes** → annotation automatique → export

---
1. Installation & configuration
2. Prompt engineering (single / entity-centric / contextuel)
3. Test manuel
4. Pipeline complet
5. Annotation automatique
6. Visualisation HTML
7. Export JSONL + CSV + CoNLL

## 1. Installation & configuration

In [ ]:
!pip install -q openai pandas tqdm

In [ ]:
import os, json, re, time, csv
from dataclasses import dataclass, asdict
from collections import defaultdict, Counter
from openai import OpenAI
from tqdm import tqdm
import pandas as pd
from IPython.display import display, HTML

# ── Alibaba DashScope ──────────────────────────────────────────────────────────
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY", "sk-460e71b3918840fb9bbc8b438541dd11")
MODEL             = "qwen-plus"   # qwen-turbo | qwen-plus | qwen-max | qwen2.5-72b-instruct

client = OpenAI(
    api_key  = DASHSCOPE_API_KEY,
    base_url = "https://dashscope-us.aliyuncs.com/compatible-mode/v1",
)

# ── Paramètres de génération ───────────────────────────────────────────────────
GEN_PARAMS = dict(
    temperature       = 0.7,
    top_p             = 1.0,
    max_tokens        = 1200,   # enough for 2-4 sentences
    frequency_penalty = 0.2,
)

# ── Test de connexion ──────────────────────────────────────────────────────────
def call_llm(system_prompt, user_prompt, temperature=0.7):
    t0 = time.time()
    response = client.chat.completions.create(
        model    = MODEL,
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        **GEN_PARAMS,
    )
    return {
        "text":       response.choices[0].message.content,
        "latency_ms": int((time.time() - t0) * 1000),
        "tokens":     response.usage.total_tokens,
    }

try:
    r = call_llm("You are helpful.", "Reply with exactly: OK")
    print(f"✅ Qwen API OK — model: {MODEL} — {r['latency_ms']} ms")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Qwen API OK — model: qwen-plus — 912 ms


## 2. Chargement du sous-graphe

On recharge `subgraph.csv` produit par `extract_subgraph.ipynb` et on le restructure au format triplet enrichi (avec IDs Wikidata).

In [ ]:
SUBGRAPH_CSV = "subgraph.csv"

# Mapping relations FR → EN
RELATION_EN = {
    "réalisateur":         "director",
    "acteur":              "cast member",
    "genre":               "genre",
    "date de publication": "publication date",
    "pays d'origine":      "country of origin",
}

# Types par propriété (pour l'annotation)
PROP_TYPES = {
    "P57":  "person",
    "P161": "person",
    "P136": "genre",
    "P577": "date",
    "P495": "country",
}

df = pd.read_csv(SUBGRAPH_CSV)
df["relation_en"] = df["prop_label"].map(RELATION_EN)

# Construire la liste de triplets au format enrichi
raw_triples = []
for _, row in df.iterrows():
    raw_triples.append({
        "subject": {
            "id":    row["sujet_id"],
            "label": row["sujet_label"],
            "type":  "film",
        },
        "predicate": {
            "id":    row["prop_id"],
            "label": row["relation_en"],
        },
        "object": {
            "id":    row["objet_id"],
            "label": row["objet_label"],
            "type":  PROP_TYPES.get(row["prop_id"], "entity"),
        },
    })

SUBGRAPH = {"triples": raw_triples}
print(f"✅ {len(SUBGRAPH['triples'])} triplets chargés")
print(f"   Relations: {df['relation_en'].value_counts().to_dict()}")
print()
for t in SUBGRAPH["triples"][:5]:
    print(f"  {t['subject']['label']:30s} [{t['predicate']['label']:20s}] {t['object']['label']}")

✅ 989 triplets chargés
   Relations: {'director': 215, 'country of origin': 215, 'cast member': 202, 'genre': 195, 'publication date': 162}

  La Bataille de Brest-Litovsk   [director            ] Alexandre Kott
  Les Lyonnais                   [director            ] Olivier Marchal
  Intouchables                   [director            ] Olivier Nakache
  Intouchables                   [director            ] Éric Toledano
  Kick-Ass                       [director            ] Matthew Vaughn


## 3. Prompt Engineering

Trois templates comme dans le pipeline Groq :
- **Single-triple** : 1 paragraphe par triplet
- **Entity-centric** : 1 paragraphe par film, couvre tous ses faits
- **Contextuel** : style contrôlé (encyclopedia, biography, news)

In [ ]:
# ════════════════════════════════════════════════
# SYSTEM PROMPT
# ════════════════════════════════════════════════
SYSTEM_PROMPT = """You are a knowledgeable writer producing natural, fluent encyclopedia-style text about cinema.
Your task: generate a short paragraph (2-4 sentences) that naturally incorporates the given knowledge graph facts.

Rules:
- Write in English only.
- Mention ALL provided entities naturally in flowing prose (no bullet points, no lists).
- Do NOT mention Wikidata or knowledge graphs.
- Be factually accurate and neutral.
- Entity names must appear EXACTLY as given (respect capitalization).
- Do not add facts that are not in the triplets.
- After the paragraph, output an annotation block.

Output format:
TEXT: <your generated paragraph>
ANNOTATIONS: <JSON array: [{"entity_id":"Q...","label":"...","start":N,"end":N,"type":"..."}]>"""


# ════════════════════════════════════════════════
# PROMPT 1 : Single triple
# ════════════════════════════════════════════════
def build_single_prompt(triple):
    s, p, o = triple["subject"], triple["predicate"], triple["object"]
    return (
        f"Generate a natural encyclopedia paragraph (2-3 sentences) for this knowledge graph triple:\n"
        f"- Subject  : {s['label']} (Wikidata ID: {s['id']}, type: {s.get('type','entity')})\n"
        f"- Predicate: {p['label']} (property: {p['id']})\n"
        f"- Object   : {o['label']} (Wikidata ID: {o['id']}, type: {o.get('type','entity')})\n\n"
        f'Express naturally the relation: "{s["label"]} {p["label"]} {o["label"]}".'
    )


# ════════════════════════════════════════════════
# PROMPT 2 : Entity-centric (multi-triple)
# ════════════════════════════════════════════════
def build_multi_prompt(triples, subject_label=None):
    seen, entities, lines = set(), [], []
    for t in triples:
        for k in ("subject", "object"):
            e = t[k]
            if e["id"] not in seen:
                seen.add(e["id"])
                entities.append(f"  - {e['label']} ({e['id']}, type: {e.get('type','entity')})")
        lines.append(
            f"  • {t['subject']['label']} —[{t['predicate']['label']}]→ {t['object']['label']}"
        )
    name = subject_label or triples[0]["subject"]["label"]
    return (
        f'Generate a paragraph (2-4 sentences) about "{name}" using these knowledge graph triples:\n'
        + "\n".join(lines)
        + "\n\nAll of these entities MUST appear naturally in the text:\n"
        + "\n".join(entities)
    )


# ════════════════════════════════════════════════
# PROMPT 3 : Style contextuel
# ════════════════════════════════════════════════
def build_contextual_prompt(triple, style="encyclopedia"):
    styles = {
        "encyclopedia": "a neutral, informative encyclopedia entry",
        "biography":    "a biographical introduction",
        "news":         "a brief news article excerpt",
        "review":       "a short film review excerpt",
    }
    s, p, o = triple["subject"], triple["predicate"], triple["object"]
    return (
        f"Write {styles.get(style, styles['encyclopedia'])} (2-3 sentences) "
        f"naturally mentioning:\n"
        f"Entity 1 : {s['label']} (ID: {s['id']}, type: {s.get('type','entity')})\n"
        f"Relation : {p['label']} (property: {p['id']})\n"
        f"Entity 2 : {o['label']} (ID: {o['id']}, type: {o.get('type','entity')})"
    )


# ════════════════════════════════════════════════
# PARSER de sortie
# ════════════════════════════════════════════════
def parse_output(raw):
    result = {"paragraph": raw.strip(), "annotations": []}
    if "TEXT:" in raw:
        after = raw.split("TEXT:", 1)[1]
        if "ANNOTATIONS:" in after:
            para, ann = after.split("ANNOTATIONS:", 1)
            result["paragraph"] = para.strip()
            ann = ann.strip().replace("```json", "").replace("```", "").strip()
            try:
                result["annotations"] = json.loads(ann)
            except:
                pass
        else:
            result["paragraph"] = after.strip()
    return result


print("✅ Prompts définis (single / entity-centric / contextuel)")

# Aperçu d'un prompt
sample = SUBGRAPH["triples"][0]
print("\nExemple single-triple prompt:")
print(build_single_prompt(sample))

✅ Prompts définis (single / entity-centric / contextuel)

Exemple single-triple prompt:
Generate a natural encyclopedia paragraph (2-3 sentences) for this knowledge graph triple:
- Subject  : La Bataille de Brest-Litovsk (Wikidata ID: Q23871, type: film)
- Predicate: director (property: P57)
- Object   : Alexandre Kott (Wikidata ID: Q2833792, type: person)

Express naturally the relation: "La Bataille de Brest-Litovsk director Alexandre Kott".


## 4. Test manuel sur quelques exemples

In [ ]:
# ── Test A : Single-triple (3 exemples) ─────────────────────────────────────
print("=" * 60)
print("TEST A — Single-triple generation")
print("=" * 60)

for triple in SUBGRAPH["triples"][:3]:
    s, p, o = triple["subject"], triple["predicate"], triple["object"]
    print(f"\n🔷 {s['label']} [{p['label']}] {o['label']}")
    result = call_llm(SYSTEM_PROMPT, build_single_prompt(triple))
    parsed = parse_output(result["text"])
    print(f"   ⏱  {result['latency_ms']} ms | 🪙 {result['tokens']} tokens")
    print(f"   📝 {parsed['paragraph']}")
    time.sleep(0.5)

TEST A — Single-triple generation

🔷 La Bataille de Brest-Litovsk [director] Alexandre Kott
   ⏱  5149 ms | 🪙 439 tokens
   📝 *La Bataille de Brest-Litovsk* is a French historical film directed by Alexandre Kott, a filmmaker known for his work in both cinema and television. The film dramatizes the pivotal 1918 negotiations between Soviet Russia and the Central Powers at Brest-Litovsk during World War I. Alexandre Kott brought his distinctive narrative sensibility to this politically charged historical subject.

🔷 Les Lyonnais [director] Olivier Marchal
   ⏱  3634 ms | 🪙 393 tokens
   📝 *Les Lyonnais* is a French crime thriller film directed by Olivier Marchal, who is also known for his work on other acclaimed French police dramas. Marchal brought his distinctive gritty realism and insider knowledge of law enforcement to the project, shaping the film’s tense, authentic portrayal of organized crime in Lyon.

🔷 Intouchables [director] Olivier Nakache
   ⏱  4567 ms | 🪙 431 tokens
   📝 *Int

In [ ]:
# ── Test B : Entity-centric (un film, tous ses triplets) ─────────────────────
print("=" * 60)
print("TEST B — Entity-centric (1 film = 1 paragraphe)")
print("=" * 60)

# Prend le film ayant le plus de triplets
from collections import defaultdict
groups = defaultdict(list)
for t in SUBGRAPH["triples"]:
    groups[t["subject"]["id"]].append(t)

best_film_id = max(groups, key=lambda k: len(groups[k]))
best_triples = groups[best_film_id]
film_name    = best_triples[0]["subject"]["label"]

print(f"\n🎬 Film: {film_name} ({len(best_triples)} triplets)")
for t in best_triples:
    print(f"   • [{t['predicate']['label']}] → {t['object']['label']}")

result = call_llm(SYSTEM_PROMPT, build_multi_prompt(best_triples, film_name))
parsed = parse_output(result["text"])
print(f"\n📝 Paragraphe généré:")
print(f"   {parsed['paragraph']}")
print(f"\n✅ {result['latency_ms']} ms | {result['tokens']} tokens")

TEST B — Entity-centric (1 film = 1 paragraphe)

🎬 Film: Star Wars, épisode VII : Le Réveil de la Force (64 triplets)
   • [director] → J. J. Abrams
   • [cast member] → Daniel Craig
   • [cast member] → Bill Hader
   • [cast member] → Judah Friedlander
   • [cast member] → Harrison Ford
   • [cast member] → Alec Guinness
   • [cast member] → Warwick Davis
   • [cast member] → Carrie Fisher
   • [cast member] → Michael Giacchino
   • [cast member] → Ewan McGregor
   • [cast member] → Max von Sydow
   • [cast member] → Andy Serkis
   • [cast member] → Mark Hamill
   • [cast member] → Simon Pegg
   • [cast member] → Harriet Walter
   • [cast member] → Meredith Salenger
   • [cast member] → Greg Grunberg
   • [cast member] → Frank Oz
   • [cast member] → Ken Leung
   • [cast member] → Thomas Sangster
   • [cast member] → Oscar Isaac
   • [cast member] → Gwendoline Christie
   • [cast member] → Anthony Daniels
   • [cast member] → Kenny Baker
   • [cast member] → Domhnall Gleeson
   • [cas

In [ ]:
# ── Test C : Variations de style ─────────────────────────────────────────────
print("=" * 60)
print("TEST C — Styles (encyclopedia / biography / news / review)")
print("=" * 60)

base = SUBGRAPH["triples"][0]
for style in ["encyclopedia", "biography", "news", "review"]:
    result = call_llm(SYSTEM_PROMPT, build_contextual_prompt(base, style))
    parsed = parse_output(result["text"])
    print(f"\n🎨 [{style.upper()}]")
    print(f"   {parsed['paragraph']}")
    time.sleep(0.5)

TEST C — Styles (encyclopedia / biography / news / review)

🎨 [ENCYCLOPEDIA]
   La Bataille de Brest-Litovsk is a historical film depicting the 1918 battle between German and Soviet forces during the Russian Civil War. The film was directed by Alexandre Kott, a Russian filmmaker known for his work in historical and political cinema. It premiered to notable attention for its ambitious reconstruction of early 20th-century military conflict.

🎨 [BIOGRAPHY]
   Alexandre Kott is a Russian film director best known for his 2017 historical war film *La Bataille de Brest-Litovsk*, which dramatizes the pivotal 1918 siege during the Russian Civil War. His direction of the film showcased a distinctive visual style and deep engagement with early Soviet history. *La Bataille de Brest-Litovsk* marked a significant entry in Kott’s filmography, reflecting both his narrative ambition and historical sensibility.

🎨 [NEWS]
   The 1938 Soviet historical drama *La Bataille de Brest-Litovsk* dramatizes the p

## 5. Annotateur automatique (rule-based)

In [ ]:
@dataclass
class EntitySpan:
    entity_id:    str
    label:        str
    entity_type:  str
    surface_form: str
    start:        int
    end:          int
    confidence:   float = 1.0
    def to_dict(self): return asdict(self)


def rule_annotate(text, entities):
    """
    Annotation automatique par matching exact + alias.
    - Matching exact du label
    - Pour les personnes : nom de famille seul
    - Pour les pays/genres : label complet
    """
    spans, seen = [], set()
    for e in entities:
        candidates = [e["label"]]
        toks = e["label"].split()
        if e.get("type") == "person" and len(toks) > 1:
            candidates += [toks[-1], toks[0]]  # nom et prénom seuls
        elif e.get("type") in ("organization", "film") and len(toks) > 1:
            candidates.append(toks[0])         # premier mot
        for surf in candidates:
            for m in re.finditer(r"\b" + re.escape(surf) + r"\b", text, re.IGNORECASE):
                k = (e["id"], m.start(), m.end())
                if k not in seen:
                    seen.add(k)
                    spans.append(EntitySpan(
                        entity_id    = e["id"],
                        label        = e["label"],
                        entity_type  = e.get("type", "entity"),
                        surface_form = text[m.start():m.end()],
                        start        = m.start(),
                        end          = m.end(),
                        confidence   = 1.0 if text[m.start():m.end()] == e["label"] else 0.85
                    ))
    return sorted(spans, key=lambda s: s.start)


def format_annotated(text, spans):
    """Affiche le texte avec les entités : [label|QID]"""
    result, cursor = [], 0
    for s in sorted(spans, key=lambda s: (s.start, -(s.end - s.start))):
        if s.start < cursor: continue
        result += [text[cursor:s.start], f"[{text[s.start:s.end]}|{s.entity_id}]"]
        cursor = s.end
    return "".join(result + [text[cursor:]])


def spans_to_iob(text, spans):
    """Convertit les spans en tags IOB2 (format CoNLL)."""
    toks = [(m.group(), m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    tags = ["O"] * len(toks)
    for s in spans:
        for i, (tok, ts, te) in enumerate(toks):
            if ts >= s.start and te <= s.end:
                tags[i] = f"B-{s.entity_type.upper()}" if ts == s.start else f"I-{s.entity_type.upper()}"
    return [(tok, tag) for (tok, _, _), tag in zip(toks, tags)]


# Test
test_txt  = "Inception is a science fiction film directed by Christopher Nolan, released in the United States in 2010."
test_ents = [
    {"id": "Q134773", "label": "Inception",         "type": "film"},
    {"id": "P57",     "label": "Christopher Nolan", "type": "person"},
    {"id": "Q183",    "label": "United States",      "type": "country"},
]
test_spans = rule_annotate(test_txt, test_ents)
print("✅ Annotateur OK:")
print(format_annotated(test_txt, test_spans))
print("\nIOB2:")
for tok, tag in spans_to_iob(test_txt, test_spans):
    if tag != "O":
        print(f"  {tok:25s} {tag}")

✅ Annotateur OK:
[Inception|Q134773] is a science fiction film directed by [Christopher Nolan|P57], released in the [United States|Q183] in 2010.

IOB2:
  Inception                 B-FILM
  Christopher               B-PERSON
  United                    B-COUNTRY
  States                    I-COUNTRY


## 6. Pipeline complet

Deux modes :
- `single` : 1 paragraphe par triplet
- `entity-centric` : 1 paragraphe par film (recommandé pour SynthIE)

In [ ]:
def run_pipeline(triples, mode="entity-centric", delay=0.5, verbose=True, max_triplets_per_film=5):
    """
    Génère des paragraphes et annote automatiquement.

    Args:
        triples              : liste de triplets KG
        mode                 : 'single' | 'entity-centric'
        delay                : secondes entre appels API
        verbose              : afficher la progression
        max_triplets_per_film: limite les triplets par film (mode entity-centric)

    Returns:
        corpus : liste de records annotés
    """
    corpus = []

    if mode == "entity-centric":
        groups = defaultdict(list)
        for t in triples:
            groups[t["subject"]["id"]].append(t)
        # Filtrer les films avec au moins 2 triplets
        items = [(k, v[:max_triplets_per_film]) for k, v in groups.items() if len(v) >= 2]
    else:
        items = list(enumerate(triples))

    total = len(items)
    print(f"🚀 Pipeline — mode: {mode} | {total} items à générer\n" + "─" * 60)

    for idx, (key, item) in enumerate(items, 1):
        try:
            # ── Prépare le prompt et les entités ─────────────────────────────
            if mode == "entity-centric":
                lbl    = item[0]["subject"]["label"]
                prompt = build_multi_prompt(item, lbl)
                seen, entities = set(), []
                for t in item:
                    for k2 in ("subject", "object"):
                        e = t[k2]
                        if e["id"] not in seen:
                            seen.add(e["id"])
                            entities.append({"id": e["id"], "label": e["label"], "type": e.get("type", "entity")})
                record_id  = f"film_{idx:04d}"
                triple_ref = item
            else:
                t      = item
                prompt = build_single_prompt(t)
                entities = [
                    {"id": t["subject"]["id"], "label": t["subject"]["label"], "type": t["subject"].get("type", "entity")},
                    {"id": t["object"]["id"],  "label": t["object"]["label"],  "type": t["object"].get("type", "entity")},
                ]
                record_id  = f"triple_{idx:04d}"
                triple_ref = t

            # ── Appel LLM ────────────────────────────────────────────────────
            result = call_llm(SYSTEM_PROMPT, prompt)
            parsed = parse_output(result["text"])

            # ── Annotation automatique ───────────────────────────────────────
            spans = rule_annotate(parsed["paragraph"], entities)

            corpus.append({
                "id":             record_id,
                "mode":           mode,
                "triple":         triple_ref,
                "paragraph":      parsed["paragraph"],
                "entities":       entities,
                "annotations":    [s.to_dict() for s in spans],
                "annotated_text": format_annotated(parsed["paragraph"], spans),
                "iob":            spans_to_iob(parsed["paragraph"], spans),
                "latency_ms":     result["latency_ms"],
                "tokens":         result["tokens"],
            })

            if verbose:
                lbl_str = (
                    item[0]["subject"]["label"] if mode == "entity-centric"
                    else f"{item['subject']['label']} → {item['object']['label']}"
                )
                print(f"[{idx:>3}/{total}] ✓ {lbl_str:40s} | {len(spans)} spans | {result['latency_ms']}ms")

        except Exception as e:
            print(f"[{idx:>3}/{total}] ✗ Erreur: {e}")

        time.sleep(delay)

    print("─" * 60)
    total_spans  = sum(len(r["annotations"]) for r in corpus)
    total_tokens = sum(r["tokens"] for r in corpus)
    print(f"\n✅ {len(corpus)} records | {total_spans} spans | {total_tokens} tokens")
    return corpus

print("✅ Pipeline défini")

✅ Pipeline défini


In [ ]:
# ── Lancement ────────────────────────────────────────────────────────────────
# mode = 'entity-centric' → 1 paragraphe riche par film (recommandé)
# mode = 'single'         → 1 paragraphe par triplet individuel

corpus = run_pipeline(
    triples              = SUBGRAPH["triples"],
    mode                 = "entity-centric",
    delay                = 0.5,
    max_triplets_per_film = 5,
)

🚀 Pipeline — mode: entity-centric | 138 items à générer
────────────────────────────────────────────────────────────
[  1/138] ✓ Les Lyonnais                             | 20 spans | 8610ms
[  2/138] ✓ Intouchables                             | 16 spans | 6952ms
[  3/138] ✓ Kick-Ass                                 | 15 spans | 8821ms
[  4/138] ✓ A Fantastic Fear of Everything           | 13 spans | 7489ms
[  5/138] ✓ Love Next Door                           | 11 spans | 9301ms
[  6/138] ✓ Skyfall                                  | 8 spans | 6040ms
[  7/138] ✓ We Are Four Lions                        | 8 spans | 7920ms
[  8/138] ✓ Star Wars, épisode VII : Le Réveil de la Force | 20 spans | 9564ms
[  9/138] ✓ Pirates 2 : La Vengeance de Stagnetti    | 6 spans | 7587ms
[ 10/138] ✓ Le Dernier Maître de l'air               | 13 spans | 10670ms
[ 11/138] ✓ Plumíferos                               | 7 spans | 6364ms
[ 12/138] ✓ Marmaduke                                | 9 spans | 9660ms
[ 13/

## 7. Visualisation HTML

In [ ]:
COLORS = {
    "film":         "#F5C4B3",
    "person":       "#d4edda",
    "country":      "#f8d7da",
    "genre":        "#fff3cd",
    "date":         "#e2d9f3",
    "organization": "#d1ecf1",
    "entity":       "#f0f0f0",
}

def render_html(record):
    text  = record["paragraph"]
    spans = sorted(record["annotations"], key=lambda s: (s["start"], -(s["end"] - s["start"])))
    result, cursor, seen = [], 0, set()
    for s in spans:
        if s["start"] < cursor: continue
        if (s["entity_id"], s["start"]) in seen: continue
        seen.add((s["entity_id"], s["start"]))
        result.append(text[cursor:s["start"]])
        color = COLORS.get(s["entity_type"], "#f0f0f0")
        result.append(
            f'<mark style="background:{color};border-radius:3px;padding:1px 5px;font-weight:600"'
            f' title="{s["entity_id"]} ({s["entity_type"]})">'
            f'{text[s["start"]:s["end"]]}'
            f'<sup style="font-size:.65em;color:#555">{s["entity_id"]}</sup></mark>'
        )
        cursor = s["end"]
    result.append(text[cursor:])
    return "".join(result)

parts = ['<style>div.r{margin:10px 0;padding:14px;border:1px solid #ddd;border-radius:8px;font-family:sans-serif;line-height:1.8;font-size:14px}</style>']
for r in corpus:
    t = r["triple"]
    if isinstance(t, list):
        title = t[0]["subject"]["label"]
        facts  = " | ".join(f"[{x['predicate']['label']}] {x['object']['label']}" for x in t)
    else:
        title = f"{t['subject']['label']} —[{t['predicate']['label']}]→ {t['object']['label']}"
        facts  = ""
    parts.append(
        f'<div class="r">'
        f'<div style="font-size:.78em;color:#555;font-weight:bold;margin-bottom:4px">🎬 {title}</div>'
        f'<div style="font-size:.72em;color:#999;margin-bottom:6px">{facts}</div>'
        f'<div>{render_html(r)}</div>'
        f'<div style="font-size:.72em;color:#aaa;margin-top:6px">'
        f'⏱ {r["latency_ms"]}ms · 🪙 {r["tokens"]} tokens · 🏷 {len(r["annotations"])} spans'
        f'</div></div>'
    )
display(HTML("\n".join(parts)))

# Stats
tc = Counter(s["entity_type"] for r in corpus for s in r["annotations"])
print("\n📊 Types d'entités annotées :")
for et, n in tc.most_common():
    print(f"  {et:15s} {'█' * min(n, 40)} {n}")


📊 Types d'entités annotées :
  person          ████████████████████████████████████████ 730
  film            ████████████████████████████████████████ 394
  genre           ████████████████████████████████████████ 192
  country         ████████████████████████████████████████ 183
  date            ████████████████████████████████████████ 54


## 8. Export JSONL + CSV + CoNLL

In [ ]:
import os
os.makedirs("output", exist_ok=True)

JSONL_PATH  = "output/wiki_cinema_synthetic.jsonl"
CSV_PATH    = "output/wiki_cinema_synthetic.csv"
CONLL_PATH  = "output/wiki_cinema_synthetic.conll"

# ── JSONL (format SynthIE) ────────────────────────────────────────────────────
with open(JSONL_PATH, "w", encoding="utf-8") as f:
    for r in corpus:
        t = r["triple"]
        if isinstance(t, list):
            triplets_clean = [{"subject": x["subject"]["label"], "relation": x["predicate"]["label"], "object": x["object"]["label"]} for x in t]
        else:
            triplets_clean = [{"subject": t["subject"]["label"], "relation": t["predicate"]["label"], "object": t["object"]["label"]}]
        record = {
            "id":             r["id"],
            "triplets":       triplets_clean,
            "text":           r["paragraph"],
            "annotated_text": r["annotated_text"],
            "annotations":    r["annotations"],
            "model":          MODEL,
            "tokens":         r["tokens"],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# ── CSV ───────────────────────────────────────────────────────────────────────
rows = []
for r in corpus:
    t = r["triple"]
    if isinstance(t, list):
        film      = t[0]["subject"]["label"]
        film_id   = t[0]["subject"]["id"]
        triplets  = " | ".join(f"{x['predicate']['label']}:{x['object']['label']}" for x in t)
        n_triplets = len(t)
    else:
        film      = t["subject"]["label"]
        film_id   = t["subject"]["id"]
        triplets  = f"{t['predicate']['label']}:{t['object']['label']}"
        n_triplets = 1
    rows.append({
        "id":             r["id"],
        "film":           film,
        "film_id":        film_id,
        "n_triplets":     n_triplets,
        "triplets":       triplets,
        "generated_text": r["paragraph"],
        "annotated_text": r["annotated_text"],
        "n_annotations":  len(r["annotations"]),
        "model":          MODEL,
        "tokens":         r["tokens"],
        "latency_ms":     r["latency_ms"],
    })

result_df = pd.DataFrame(rows)
result_df.to_csv(CSV_PATH, index=False, encoding="utf-8")

# ── CoNLL IOB2 ────────────────────────────────────────────────────────────────
with open(CONLL_PATH, "w", encoding="utf-8") as f:
    for r in corpus:
        f.write(f"# {r['id']}\n")
        for tok, tag in r["iob"]:
            f.write(f"{tok}\t{tag}\n")
        f.write("\n")

print(f"✅ JSONL  → {JSONL_PATH}  ({os.path.getsize(JSONL_PATH)//1024} Ko)")
print(f"✅ CSV    → {CSV_PATH}  ({os.path.getsize(CSV_PATH)//1024} Ko)")
print(f"✅ CoNLL  → {CONLL_PATH}  ({os.path.getsize(CONLL_PATH)} octets)")
print(f"\n📊 {len(result_df)} exemples | moy. {result_df['n_triplets'].mean():.1f} triplets/texte")
print(f"   Tokens total: {result_df['tokens'].sum()} | moy. {result_df['tokens'].mean():.0f}/texte")
result_df[["film", "n_triplets", "n_annotations", "generated_text"]].head(10)

✅ JSONL  → output/wiki_cinema_synthetic.jsonl  (439 Ko)
✅ CSV    → output/wiki_cinema_synthetic.csv  (169 Ko)
✅ CoNLL  → output/wiki_cinema_synthetic.conll  (99454 octets)

📊 138 exemples | moy. 3.6 triplets/texte
   Tokens total: 90172 | moy. 653/texte


,film,n_triplets,n_annotations,generated_text
0,Les Lyonnais,5,20,"*Les Lyonnais*, a French crime drama released ..."
1,Intouchables,5,16,"*Intouchables*, the acclaimed 2011 French film..."
2,Kick-Ass,5,15,Kick-Ass is a vibrant and irreverent film de s...
3,A Fantastic Fear of Everything,4,13,A Fantastic Fear of Everything is a 2012 coméd...
4,Love Next Door,4,11,*Love Next Door* is a film directed by Julian ...
5,Skyfall,2,8,"Skyfall, the twenty-third installment in the J..."
6,We Are Four Lions,3,8,"*We Are Four Lions*, a satirical black comedy ..."
7,"Star Wars, épisode VII : Le Réveil de la Force",5,20,"Star Wars, épisode VII : Le Réveil de la Force..."
8,Pirates 2 : La Vengeance de Stagnetti,3,6,"*Pirates 2 : La Vengeance de Stagnetti*, a fil..."
9,Le Dernier Maître de l'air,5,13,"Le Dernier Maître de l'air, un film américain ..."


In [ ]:
# ── Aperçu final des textes générés ──────────────────────────────────────────
print("=== Échantillon aléatoire du dataset ===")
sample = result_df.sample(min(5, len(result_df)), random_state=42)
for _, row in sample.iterrows():
    print(f"\n🎬 {row['film']}  ({row['n_triplets']} triplets → {row['n_annotations']} entités annotées)")
    print(f"   Triplets  : {row['triplets']}")
    print(f"   Texte     : {row['generated_text']}")
    print(f"   Annoté    : {row['annotated_text']}")

=== Échantillon aléatoire du dataset ===

🎬 Thale  (2 triplets → 10 entités annotées)
   Triplets  : director:Aleksander L. Nordaas | country of origin:Norvège
   Texte     : Thale is a Norwegian folk horror film that marks the feature debut of writer-director Aleksander L. Nordaas. Released in 2012, the film was produced in Norvège and draws on regional folklore to craft its atmospheric, minimalist narrative. Thale stands out for its distinctive visual style and restrained storytelling, establishing Aleksander L. Nordaas as a distinctive voice in contemporary Norvège cinema.
   Annoté    : [Thale|Q82847] is a Norwegian folk horror film that marks the feature debut of writer-director [Aleksander L. Nordaas|Q16221947]. Released in 2012, the film was produced in [Norvège|Q20] and draws on regional folklore to craft its atmospheric, minimalist narrative. [Thale|Q82847] stands out for its distinctive visual style and restrained storytelling, establishing [Aleksander L. Nordaas|Q16221947] a